In [ ]:
import os
import numpy as np
from os.path import join
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import jaccard_score
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import pairwise_distances
import seaborn as sns 

In [ ]:
from datasets import load_dataset, DatasetDict, concatenate_datasets
from transformers import AutoTokenizer

def load_datasets() -> tuple[DatasetDict, list[str]]:
    """Loads and splits LeNER-Br and ASSIN2 datasets for NER task."""

    # Load LeNER-Br from Hugging Face
    raw_ner = load_dataset("peluz/lener_br", revision="refs/convert/parquet")
    ner_ds = DatasetDict({
        "train": raw_ner["train"],
        "validation": raw_ner["validation"],
        "test": raw_ner["test"]
    })
    
    # Get the unique list of NER tags
    label_list = ner_ds["train"].features["ner_tags"].feature.names
    
    return ner_ds, label_list

def preprocess_ner(examples: dict, tokenizer: AutoTokenizer) -> dict:
    """Tokenizes word sequences and aligns NER tags with the resulting tokens."""
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        return_overflowing_tokens=True
    )
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            # Map special tokens to -100 to ignore them in loss
            if word_idx is None:
                label_ids.append(-100)
            # Assign label only to the first token of a split word
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(label[word_idx])
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

def get_all_sentences_of_split(dataset:DatasetDict, split:str) -> dict:
    if split == "dev":
        split = "validation"
    ner = dataset[split].to_pandas()
    return {" ".join(list(k)):(v,i) for k,v,i in zip(ner["tokens"].to_list(), ner["ner_tags"].to_list(), ner["id"].to_list())}

def get_all_documents_of_split(split:str):
    docs = {}
    for doc in os.listdir(join("leNER-Br", split)):
        if doc == f"{split}.conll":
            continue
        with open(join("leNER-Br", split, doc)) as f:
            content = f.read().splitlines()
            content = list(map(lambda x: x.split()[0] if len(x) > 1 else "", content))
        doc_content = ""
        for token in content:
            doc_content += f"{token} "
        docs[doc_content] = doc
    return docs

def calculate_sim_matrix(docs1:dict, docs2:dict):
    docs1_to_sim = list(docs1.keys())
    docs2_to_sim = list(docs2.keys())

    # 1. Fit once on all documents
    vectorizer = CountVectorizer(binary=True, token_pattern=r"\S+")
    vectorizer.fit(docs1_to_sim + docs2_to_sim)

    X_train = vectorizer.transform(docs1_to_sim).toarray().astype(bool)
    X_test = vectorizer.transform(docs2_to_sim).toarray().astype(bool)

    # 2. Compute pairwise Jaccard distance (1 - similarity)
    jaccard_dist = pairwise_distances(X_train, X_test, metric='jaccard', n_jobs=14)
    sim_matrix = 1.0 - jaccard_dist

    max_sim = np.max(sim_matrix)

    return sim_matrix, max_sim, np.where(sim_matrix == max_sim)

def group_bi(row):
    mapping = {0:0,1:1,2:1,3:2,4:2,5:3,6:3,7:4,8:4,9:5,10:5,11:6,12:6}
    row["ner_tags"] = list(map(lambda x: mapping[x], row["ner_tags"]))
    return row

In [ ]:
test = os.listdir(join("leNER-Br", "test"))
train = os.listdir(join("leNER-Br", "train"))
raw_text = os.listdir(join("leNER-Br", "raw_text"))
raw_text = list(map(lambda x: x.split(".")[0], raw_text))

In [ ]:
missing_files = []
for doc in test:
    if doc.split(".")[0] not in raw_text:
        missing_files.append(doc)
missing_files

In [ ]:
missing_files = []
for doc in train:
    if doc.split(".")[0] not in raw_text:
        missing_files.append(doc)
missing_files

In [ ]:
cont = 0
for doc in test:
    with open(join("leNER-Br", "test", doc)) as f:
        content = f.read().splitlines()
    if cont == 1:
        print(content, doc)
    cont += 1

In [ ]:
print(test)
print(train)

In [ ]:
entities = set()
for doc in train:
    with open(join("leNER-Br", "train", doc)) as f:
        content = f.read().splitlines()
    for line in content:
        if len(line.split()) > 0:
            entities.add(line.split()[1].split("-")[1] if "-" in line.split()[1] else line.split()[1])
entities

In [ ]:
entities_dicts = {
    'JURISPRUDENCIA':{"train":[], "test":[]},
    'LEGISLACAO':{"train":[], "test":[]},
    'LOCAL':{"train":[], "test":[]},
    'O':{"train":[], "test":[]},
    'ORGANIZACAO':{"train":[], "test":[]},
    'PESSOA':{"train":[], "test":[]},
    'TEMPO':{"train":[], "test":[]}
}
# for doc in train:
with open(join("leNER-Br", "train", "train.conll")) as f:
    content = f.read().splitlines()
for line in content:
    if len(line) > 1:
        subtext, curr_entity = line.split()
        entities_dicts[curr_entity.split("-")[1] if curr_entity != "O" else "O"]["train"].append(subtext)
# for doc in test:
with open(join("leNER-Br", "test", "test.conll")) as f:
    content = f.read().splitlines()
for line in content:
    if len(line) > 1:
        subtext, curr_entity = line.split()
        entities_dicts[curr_entity.split("-")[1] if curr_entity != "O" else "O"]["test"].append(subtext)

In [ ]:
total_train = {
    'JURISPRUDENCIA':0,
    'LEGISLACAO':0,
    'LOCAL':0,
    'ORGANIZACAO':0,
    'PESSOA':0,
    'TEMPO':0
}
total_test = {
    'JURISPRUDENCIA':0,
    'LEGISLACAO':0,
    'LOCAL':0,
    'ORGANIZACAO':0,
    'PESSOA':0,
    'TEMPO':0
}

In [ ]:
for entity in sorted(entities):
    if entity == "O":
        continue
    print(entity)
    subtext_counts_train = {}
    subtext_counts_test = {}
    for subtext in tqdm(entities_dicts[entity]["train"]):
        subtext_counts_train[subtext] = entities_dicts[entity]["train"].count(subtext)
        total_train[entity] += 1
    for subtext in tqdm(entities_dicts[entity]["test"]):
        subtext_counts_test[subtext] = entities_dicts[entity]["test"].count(subtext)
        total_test[entity] += 1
    entities_dicts[entity]["train"] = subtext_counts_train
    entities_dicts[entity]["test"] = subtext_counts_test

with open("./frequencies.txt", "w", encoding="utf-8") as f:
    for entity in sorted(entities):
        if entity == "O":
            continue
        entities_dicts[entity]["train"] = dict(sorted(entities_dicts[entity]["train"].items(), key=lambda item: item[1], reverse=True))
        entities_dicts[entity]["test"] = dict(sorted(entities_dicts[entity]["test"].items(), key=lambda item: item[1], reverse=True))
        f.write(f"{entity}\n")
        f.write("TRAIN\n")
        for subtext in entities_dicts[entity]["train"]:
            f.write(f'{subtext} - {entities_dicts[entity]["train"][subtext]}\n')
        f.write("TEST\n")
        for subtext in entities_dicts[entity]["test"]:
            f.write(f'{subtext} - {entities_dicts[entity]["test"][subtext]}\n')
        f.write("\n")

In [ ]:
total_train_leak = {
    'JURISPRUDENCIA':0,
    'LEGISLACAO':0,
    'LOCAL':0,
    'ORGANIZACAO':0,
    'PESSOA':0,
    'TEMPO':0
}

total_test_leak = {
    'JURISPRUDENCIA':0,
    'LEGISLACAO':0,
    'LOCAL':0,
    'ORGANIZACAO':0,
    'PESSOA':0,
    'TEMPO':0
}

with open("leak.txt", "w", encoding="utf-8") as f:
    for entity in total_train_leak.keys():
        cont = 1
        for subtext in entities_dicts[entity]["train"].keys():
            if subtext in entities_dicts[entity]["test"].keys():
                f.write(f"{entity}\n")
                f.write(f'{cont}    {subtext} - TRAIN: {entities_dicts[entity]["train"][subtext]} - TEST: {entities_dicts[entity]["test"][subtext]}\n\n')
                total_train_leak[entity] += entities_dicts[entity]["train"][subtext]
                total_test_leak[entity] += entities_dicts[entity]["test"][subtext]
                cont += 1
        f.write(f"{entity}\n")
        f.write(f'TOTAL - TRAIN: {total_train_leak[entity]} ({round(100 * total_train_leak[entity] / total_train[entity],2)}%) - TEST: {total_test_leak[entity]} ({round(100 * total_test_leak[entity] / total_test[entity],2)}%)\n\n')
        f.write("\n\n")

In [ ]:
all_entities_train = sum(total_train.values())
all_entities_test = sum(total_test.values())
all_leak_train = sum(total_train_leak.values())
all_leak_test = sum(total_test_leak.values())

percentual_train_leak = total_train_leak.copy()
for entity in percentual_train_leak.keys():
    percentual_train_leak[entity] = round(100 * total_train_leak[entity] / total_train[entity],2)
x = list(percentual_train_leak.keys())
y = percentual_train_leak.values()
_, ax = plt.subplots(1,2,figsize=(20,8))
ax[0].set_ylim(0,100)
ax[0].set_title("Tokens Vazados por Classe no Treino (%)")
bars = ax[0].bar(x,y)
ax[0].bar_label(bars)

percentual_test_leak = total_test_leak.copy()
for entity in percentual_test_leak.keys():
    percentual_test_leak[entity] = round(100 * total_test_leak[entity] / total_test[entity],2)
x = list(percentual_test_leak.keys())
y = percentual_test_leak.values()
ax[1].set_ylim(0,100)
ax[1].set_title("Tokens Vazados Usados no Teste (%)")
bars = ax[1].bar(x,y)
ax[1].bar_label(bars)

plt.show()

In [ ]:
total_train_pseudo_leak = {
    'JURISPRUDENCIA':0,
    'LEGISLACAO':0,
    'LOCAL':0,
    'ORGANIZACAO':0,
    'PESSOA':0,
    'TEMPO':0
}

total_test_leak = {
    'JURISPRUDENCIA':0,
    'LEGISLACAO':0,
    'LOCAL':0,
    'ORGANIZACAO':0,
    'PESSOA':0,
    'TEMPO':0
}

with open("pseudo_leaks.txt", "w", encoding="utf-8") as f:
    for entity in total_train_pseudo_leak.keys():
        cont = 1
        for subtext in entities_dicts[entity]["train"].keys():
            matches_count = 1
            if subtext in entities_dicts[entity]["test"].keys():
                for aux_entity in total_train_pseudo_leak.keys():
                    if aux_entity == entity:
                        continue
                # f.write(f"{entity}\n")
                # f.write(f'{cont}    {subtext} - TRAIN: {entities_dicts[entity]["train"][subtext]} - TEST: {entities_dicts[entity]["test"][subtext]}\n\n')
                # total_train_pseudo_leak[entity] += entities_dicts[entity]["train"][subtext]
                # total_test_leak[entity] += entities_dicts[entity]["test"][subtext]
                # cont += 1
        f.write(f"{entity}\n")
        f.write(f'TOTAL - TRAIN: {total_train_pseudo_leak[entity]} - TEST: {total_test_leak[entity]}\n\n')
        f.write("\n\n")

In [ ]:
dataset, ner_classes = load_datasets()

In [ ]:
dataset.save_to_disk("leNER-Br_v1")

In [ ]:
dataset["train"] = dataset["train"].map(group_bi)
dataset["test"] = dataset["test"].map(group_bi)
dataset["validation"] = dataset["validation"].map(group_bi)

In [ ]:
train_sentences = get_all_sentences_of_split(dataset, "train")
test_sentences = get_all_sentences_of_split(dataset, "test")
dev_sentences = get_all_sentences_of_split(dataset, "dev")

In [ ]:
all_test_sentences = {}
for doc in test:
    sentence = ""
    with open(join("leNER-Br", "test", doc)) as f:
        content = f.read().splitlines()
    for word in content:
        if len(word) > 0:
            sentence += f"{word.split()[0]} "
        else:
            sentence = sentence[:len(sentence)-1:]
            if sentence not in all_test_sentences.keys():
                all_test_sentences[sentence] = []
            all_test_sentences[sentence].append(doc)
            sentence = ""

In [ ]:
print(len(sentence_pos_train_test[0]))
print(len(sentence_pos_train_dev[0]))
print(len(sentence_pos_test_dev[0]))

In [ ]:
leaked_sentences = {
    "train":{},
    "test":{},
    "dev":{}
}

In [ ]:
for train_pos, test_pos in zip(sentence_pos_train_test[0], sentence_pos_train_test[1]):
    leaked_sentences["train"][train_sentences[list(train_sentences.keys())[train_pos]][1]] = (train_sentences[list(train_sentences.keys())[train_pos]][0], list(train_sentences.keys())[train_pos])
    leaked_sentences["test"][test_sentences[list(test_sentences.keys())[test_pos]][1]] = (test_sentences[list(test_sentences.keys())[test_pos]][0], list(test_sentences.keys())[test_pos])

In [ ]:
for train_pos, dev_pos in zip(sentence_pos_train_dev[0], sentence_pos_train_dev[1]):
    leaked_sentences["train"][train_sentences[list(train_sentences.keys())[train_pos]][1]] = (train_sentences[list(train_sentences.keys())[train_pos]][0], list(train_sentences.keys())[train_pos])
    leaked_sentences["dev"][dev_sentences[list(dev_sentences.keys())[dev_pos]][1]] = (dev_sentences[list(dev_sentences.keys())[dev_pos]][0], list(dev_sentences.keys())[dev_pos])

In [ ]:
print(len(leaked_sentences["train"]))
print(len(leaked_sentences["test"]))
print(len(leaked_sentences["dev"]))

In [ ]:
groupings = {
    k:{"train":len(entities[k]["train"]), "test":len(entities[k]["test"]), "validation":len(entities[k]["validation"])} for k in entities.keys()
}
groupings

In [ ]:
groupings = pd.DataFrame(groupings)
groupings

In [ ]:
groupings.T.plot(kind="bar")

plt.xlabel("")
plt.ylabel("Valor")
plt.legend(title="Categoria")
plt.tight_layout()
plt.show()

In [ ]:
tokens_labels = {}
splits = ["train", "test", "validation"]
for i in range(1,7):
    for split in splits:
        for token in entities[i][split]:
            if token not in tokens_labels.keys():
                tokens_labels[token] = {"train":{index:0 for index in range(1,7)}, "test":{index:0 for index in range(1,7)}, "validation":{index:0 for index in range(1,7)}}
            tokens_labels[token][split][i] += 1

In [ ]:
tokens_labels = dict(sorted(tokens_labels.items(), key=lambda item: sum(item[1]["train"].values()), reverse=True))

In [ ]:
ner_ds

In [ ]:
ner_ds["train"] = ner_ds["train"].filter(lambda x: x["id"] not in leaked_sentences["train"])
ner_ds["validation"] = ner_ds["validation"].filter(lambda x: x["id"] not in leaked_sentences["dev"])

In [ ]:
ner_ds.save_to_disk("leNER-Br_filtered")

In [ ]:
ner_ds

In [ ]:
lengths = []
for sentence in sentence_pos_train_test[0]:
    lengths.append(len(list(train_sentences.keys())[sentence]))
for sentence in sentence_pos_train_dev[0]:
    lengths.append(len(list(train_sentences.keys())[sentence]))

sns.histplot(lengths, bins=50, alpha=0.5)
plt.title("Leaked Train Sentences Sizes")
plt.ylim(0,65)
plt.xlim(0,525)
plt.show()

In [ ]:
lengths = []
for sentence in sentence_pos_train_test[1]:
    lengths.append(len(list(test_sentences.keys())[sentence]))
for sentence in sentence_pos_test_dev[0]:
    lengths.append(len(list(test_sentences.keys())[sentence]))

sns.histplot(lengths, bins=50, alpha=0.5)
plt.title("Leaked Test Sentences Sizes")
plt.ylim(0,65)
plt.xlim(0,525)
plt.show()

In [ ]:
cont = 0
for row in sim_matrix:
    cont += list(row).count(1)
cont

In [ ]:
cont = 0
for sentence in train_docs:
    if sentence in test_docs:
        if cont < 15:
            print(sentence)
        else:
            break
        cont += 1

In [ ]:
import seaborn as sns 


sns.histplot(sim_matrix.max(axis=0), bins=50, alpha=0.5)
